
## **CSIS 1230 — Week 3 Monday**  
### **Inheritance Hierarchies & Method Overriding**


**Today’s goals**
- Understand *inheritance* and when to use it
- Build and reason about *inheritance hierarchies*
- Use `super()` and implement *method overriding*
- Recognize best practices and common pitfalls
- Apply concepts in short exercises and a mini‑project

 **_Context:_** 
 - Week 1 you created classes with constructors, `__str__`/`__repr__`, and you modeled a `Course` in the Friday Lab. 
 - In Week 2 you practiced encapsulation (public/private) and getters/setters. 
 
 - Today we reuse and extend classes via inheritance.



### **1) What is Inheritance?**

- Inheritance lets a class (the *child* or *subclass*) reuse attributes and methods of another class (the *parent* or *superclass*).  

- It models "_is‑a_" relationships and promotes **reusability**, **extensibility**, and **maintainability**.

 **Analogy:**  


- `Professor` _is a_ `Person` with a doctorate degree

- `Student` _is a_ `Person` enrolled in school

- `Car` _is a_ type of `Vehicle`

- `Electricvehicle` _is a_ specialized `Vehicle`

**When to apply**
- Several classes share common state/behavior; keep the common parts in a base class.
- You want to refine or specialize behavior in a subclass via **overriding**.
- You need a consistent interface across different specialized types.


In [5]:

# Basic inheritance: Vehicle -> Car

class Vehicle:
    def __init__(self, make: str, model: str) -> None:
        self.make = make
        self.model = model

    def drive(self) -> str:
        return f"{self.make} {self.model} is driving."

class Car(Vehicle):
    def __init__(self, make: str, model: str, doors: int) -> None:
        # Use super() to initialize the inherited part
        super().__init__(make, model)
        self.doors = doors

    def honk(self) -> str:
        return "Beep beep!"

v = Vehicle("Generic", "Transport")
c = Car("Toyota", "Camry", 4)

print(v.drive())   # inherited method lives in Vehicle
print(c.drive())   # Car inherits drive() ### Due to inhertance, we can call it like this 
print(c.honk())    # Car-specific behavior

# class SUV(Vehicle):
#     def __init__(self, make, model, doors): ### Because Car have "doors" -> change "MotorBike" -> "SUV"
#         super().__init__(make, model, doors) ### Like this, we can make other things 

# honda_suv = SUV("Suzuki", "004")
# honda_suv.drive()

### SUMMARY 
### (drive balance) Vehicle <- (ls-a) Car <- (ls-a) SUV -> drive 

### diff version
class drive_Car(Vehicle):
    def __init__(self, make, model): ### Because Car have "doors" -> change "MotorBike" -> "SUV"
        super().__init__(make, model) ### Like this, we can make other things 

honda_suv = drive_Car("Suzuki", "004")
honda_suv.drive()

Generic Transport is driving.
Toyota Camry is driving.
Beep beep!


'Suzuki 004 is driving.'


### **2) Inheritance Hierarchies**
Classes can form multi‑level hierarchies. Keep them shallow (2–3 levels) to avoid complexity.


In [6]:

# Extending the hierarchy: Vehicle -> Car -> ElectricCar

class ElectricCar(Car):
    def __init__(self, make: str, model: str, doors: int, battery_kwh: float) -> None:
        super().__init__(make, model, doors)
        self.battery_kwh = battery_kwh

    def charge(self) -> str:
        return f"{self.make} {self.model} is charging. Battery: {self.battery_kwh} kWh"

tesla = ElectricCar("Tesla", "Model 3", 4, 75.0)
print(tesla.drive())   # from Vehicle
print(tesla.honk())    # from Car
print(tesla.charge())  # from ElectricCar


Tesla Model 3 is driving.
Beep beep!
Tesla Model 3 is charging. Battery: 75.0 kWh



### **3) Method Overriding**

- A **subclass** can redefine a method from the **parent** using the **same name**.  

- Use overriding to customize or extend behavior for the child type.


In [7]:

# Overriding example: different animals speak differently

class Animal:
    def speak(self) -> str:
        return "Some generic sound."

class Dog(Animal): 
    def speak(self) -> str:  # overrides Animal.speak
        return "Woof!"

class Cat(Animal):
    def speak(self) -> str:
        return "Meow!"

animals = [Dog(), Cat(), Animal()] ### list of animals 
for a in animals:
    print(type(a).__name__, "->", a.speak())


Dog -> Woof!
Cat -> Meow!
Animal -> Some generic sound.


In [8]:
### e.g., Method of Overriding 

class Shape:
    def area():
        return "Implement yours"
class circle(Shape):
    def __init__(self, radius):
        self.radius = radius 
    def area(self, radius):
        return 3.14 * radius **2

class Triangle(Shape):
    def area(self, base, height):
        return 0.5 * base * height


### Using `super()` to extend behavior (not just replace it)
Sometimes the child wants to do everything the parent does **plus** a little more.


In [9]:

class BankAccount:
    def __init__(self, balance: float = 0.0) -> None:
        self._balance = float(balance) ### [CAUTION]: Don't access to private 

    def withdraw(self, amount: float) -> str:
        if amount <= self._balance: ### it checks amount 
            self._balance -= amount
            return f"Withdrew ${amount:.2f}. Balance = ${self._balance:.2f}"
        return "Insufficient funds." 

class SavingsAccount(BankAccount): 
    def __init__(self, balance: float = 0.0, fee: float = 5.0) -> None:
        super().__init__(balance)
        self._fee = fee

    def withdraw(self, amount: float) -> str:
        # extend the base behavior with a fee
        total = amount + self._fee
        if total <= self._balance:
            # reuse base method for core logic if desired:
            # return super().withdraw(total)  # alternative
            self._balance -= total
            return f"Withdrew ${amount:.2f} + ${self._fee:.2f} fee. Balance = ${self._balance:.2f}"
        return "Insufficient funds (including fee)."

acct = SavingsAccount(100, fee=2.5)
print(acct.withdraw(20))
print(acct.withdraw(200))


Withdrew $20.00 + $2.50 fee. Balance = $77.50
Insufficient funds (including fee).



### **4) Best Practices for Inheritance**
- Prefer **composition** over inheritance when the relationship is "_has‑a_" rather than "_is‑a_".  
  - Example: `Car` **has a** `Battery` (composition), but `ElectricCar` **is a** `Car` (inheritance).
- Keep hierarchies **shallow** and **clear**.
- Call `super()` in `__init__` (and other overrides) to keep base behavior intact.
- Avoid changing a method’s **public signature** when overriding.
- Obey **Liskov Substitution Principle**: a `SavingsAccount` should behave like a `BankAccount` wherever it’s used.
- Don’t reach into a parent’s **private** state; use protected attributes (`_like_this`) or public methods.


In [10]:

# Composition vs. Inheritance demo

class Battery:
    def __init__(self, kwh: float) -> None:
        self.kwh = kwh

    def info(self) -> str:
        return f"{self.kwh} kWh battery"

class EV:  # Not inheriting from Battery; it *has a* Battery
    def __init__(self, make: str, model: str, battery: Battery) -> None:
        self.make = make
        self.model = model
        self.battery = battery

    def describe(self) -> str:
        return f"{self.make} {self.model} with {self.battery.info()}"

my_ev = EV("Nissan", "Leaf", Battery(40))
print(my_ev.describe())


Nissan Leaf with 40 kWh battery



### **5) Common Errors & Debug Tips**
- **Forgetting `super().__init__`**: child misses parent initialization → attributes missing.
- **Overriding with a different signature**: breaks callers expecting the parent’s interface.
- **Overusing inheritance**: when composition or helper functions would be simpler.
- **Accessing `__private` attributes** from parent directly: name‑mangling makes it fragile.
- **Deep hierarchies**: make debugging and reasoning difficult.



### Reference Solution (reveal after attempting)



### **6) `isinstance`, `issubclass`, and Method Resolution Order (MRO)**

```python 
1. isinstance(obj, Class) 
``` 
checks if an object is an instance of a class **or any of its parents**. 
``` python
2. issubclass(Child, Parent)
```  
```python 
3. MRO (method resolution order): 
```
Answers the question:
If this method exists in multiple classes, which one does Python use?
is the order Python uses to look up attributes/methods.

```python 
 Class.mro()
```

In [ ]:

print(isinstance(5, int), isinstance(True, int))  # bool is a subclass of int in Python
print(issubclass(bool, int))
print(issubclass(Car, Vehicle))
print(isinstance(my_ev, EV))

# Inspect MRO for our earlier classes
print("\nMRO examples:")
print("ElectricCar MRO:", [c.__name__ for c in ElectricCar.mro()])


### We have to check the hierarchy to check the property 



True True
True
True
True

MRO examples:
ElectricCar MRO: ['ElectricCar', 'Car', 'Vehicle', 'object']



### **7) Classwork**

 
**Part 1: Implement** ``` SavingsAccount``` <br>

- Create a SavingsAccount class that inherits from ```BankAccount```.

**Requirements:**

- It should have all the functionality of a regular BankAccount.

- It should allow up to 5 free withdrawals per month.

- After the 5th withdrawal, a fee of $2.00 should be charged for each subsequent 
withdrawal in that month.

- Implement a method reset_withdrawal_count() that sets the withdrawal counter back to zero (simulating a new month).


Here is a starter code below:

In [ ]:
class BankAccount:
    """Base class for a simple bank account."""
    def __init__(self, account_holder: str, initial_balance: float = 0.0):
        self.account_holder = account_holder
        self._balance = initial_balance  # Protected attribute

    def deposit(self, amount: float) -> str:
        if amount > 0:
            self._balance += amount
            return f"Deposited ${amount:.2f}. New balance: ${self._balance:.2f}"
        return "Deposit amount must be positive."

    def withdraw(self, amount: float) -> str:
        if amount <= 0:
            return "Withdrawal amount must be positive."
        if amount > self._balance:
            return "Insufficient funds."
        self._balance -= amount
        return f"Withdrew ${amount:.2f}. New balance: ${self._balance:.2f}"

    def get_balance(self) -> float:
        return self._balance

    def __str__(self) -> str:
        return f"Account holder: {self.account_holder}, Balance: ${self._balance:.2f}"

# Take note point: protect balance -> inheritance 
# **Part 1: Implement** ``` SavingsAccount``` <br> - Create a SavingsAccount class that inherits from ```BankAccount```.
# **Requirements:** - It should have all the functionality of a regular BankAccount.
# - It should allow up to 5 free withdrawals per month.
# - After the 5th withdrawal, a fee of $2.00 should be charged for each subsequent 
# withdrawal in that month.
# - Implement a method reset_withdrawal_count() that sets the withdrawal counter back to zero (simulating a new month).
class SavingsAccount(BankAccount):
    def __init__(self, account_holder, initial_balance=0, interest_rate=2.00):
        super.__init__(account_holder, initial_balance)
        self.interest_rate = interest_rate
        self.withdraw_monthly_count = 0 

    def deposit(self, amount: float) -> str: 
        self.withdraw_monthly_count += 1
        if self.withdraw_monthly_count >= 5:
            interest = self._balance * self.interest_rate
            self._balance = interest 
            return f"Applied interest of ${interest:.2f}. New balance: ${self._balance:.2f}"
        else:
            return f""

    def withdraw(self, amount: float, withdrawl_count) -> str:
        if withdrawl_count > 5: 
            fee += 2 
            withdrawl_count += 1 
            
    def get_balance(self) -> float:

    # def __str__(self) -> str:
        
account = SavingsAccount("Alice", 1000, 0.03)
